## Partindo do que já ficou pronto na Etapa 1 (EDA+Baseline)

Recarregar os dados e repitir o pré-processamento já validado: correção do
TotalCharges, remoção de customerID, encoding das categóricas e split
treino/teste estratificado

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report

df = pd.read_csv('../data/Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

df_model = df.drop(columns=['customerID'])
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})

categoricas = df_model.select_dtypes(include=['object', 'str']).columns
le = LabelEncoder()
for col in categoricas:
    df_model[col] = le.fit_transform(df_model[col])

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1109)

## Baseline: Regressão Logística (recapitulando da Etapa 1)

Reproduzir o baseline da Etapa 1, com os dados escalados e
class_weight='balanced', para servir de referência direta na comparação
com os modelos de árvore e rede neural deste notebook.

In [24]:
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=1109, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

print(classification_report(y_test, lr_model.predict(X_test_scaled)))

              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1035
           1       0.51      0.80      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409



## Random Forest

Testar um modelo baseado em árvores pra comparar com o baseline linear.
Random Forest costuma lidar melhor com relações não-lineares, e não
precisa dos dados escalados como a regressão logística precisava.

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(class_weight='balanced', random_state=1109)
rf_model.fit(X_train, y_train)

print(classification_report(y_test, rf_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.86      0.83      0.84      1035
           1       0.57      0.64      0.60       374

    accuracy                           0.78      1409
   macro avg       0.72      0.73      0.72      1409
weighted avg       0.79      0.78      0.78      1409



## Árvore de Decisão (Decision Tree / CART)

Testar uma árvore de decisão simples,tende a
ser menos conservadora com a classe minoritária que um ensemble.

In [26]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(class_weight='balanced', random_state=1109)
dt_model.fit(X_train, y_train)

print(classification_report(y_test, dt_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.83      0.81      0.82      1035
           1       0.51      0.55      0.53       374

    accuracy                           0.74      1409
   macro avg       0.67      0.68      0.68      1409
weighted avg       0.75      0.74      0.74      1409



## MLPClassifier (rede neural simples)

Testar uma rede neural simples do scikit-learn. 

In [27]:
from sklearn.neural_network import MLPClassifier

mlp_model = MLPClassifier(random_state=1109, max_iter=500)
mlp_model.fit(X_train_scaled, y_train)

print(classification_report(y_test, mlp_model.predict(X_test_scaled)))

              precision    recall  f1-score   support

           0       0.83      0.85      0.84      1035
           1       0.56      0.53      0.54       374

    accuracy                           0.76      1409
   macro avg       0.70      0.69      0.69      1409
weighted avg       0.76      0.76      0.76      1409



c:\Users\Pichau\mlet-fase1-churn-prediction\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


## Ajuste do MLP: mais iterações + balanceamento manual

MLPClassifier não tem class_weight, mas aceita sample_weight no fit,
calculando pesos balanceados manualmente com compute_sample_weight, igual
em espírito ao que fizemos nos outros modelos. Também aumento max_iter
para resolver o warning de convergência.

In [28]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

mlp_model = MLPClassifier(random_state=1109, max_iter=1000)
mlp_model.fit(X_train_scaled, y_train)  # MLP não aceita sample_weight no fit padrão

print(classification_report(y_test, mlp_model.predict(X_test_scaled)))

              precision    recall  f1-score   support

           0       0.83      0.86      0.85      1035
           1       0.57      0.51      0.54       374

    accuracy                           0.77      1409
   macro avg       0.70      0.69      0.69      1409
weighted avg       0.76      0.77      0.76      1409



## MLP com balanceamento (SMOTE)

Sem balanceamento, o MLP ficou com recall baixo (0,51), parecido com o
problema que tivemos no baseline original. Testar agora com SMOTE.

In [29]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=1109)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

mlp_model = MLPClassifier(random_state=1109, max_iter=1000)
mlp_model.fit(X_train_smote, y_train_smote)

print(classification_report(y_test, mlp_model.predict(X_test_scaled)))

              precision    recall  f1-score   support

           0       0.83      0.80      0.81      1035
           1       0.49      0.54      0.52       374

    accuracy                           0.73      1409
   macro avg       0.66      0.67      0.66      1409
weighted avg       0.74      0.73      0.73      1409



## Validação cruzada (Regressão Logística)

Rodar validação cruzada estratificada (5 folds) no modelo campeão, pra
confirmar que o recall de 0,80 não foi só sorte de um split específico, 
e sim um resultado consistente do modelo.

In [30]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

scaler_cv = StandardScaler()
X_scaled = scaler_cv.fit_transform(X)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1109)

final_model = LogisticRegression(max_iter=1000, random_state=1109, class_weight='balanced')
scores = cross_val_score(final_model, X_scaled, y, cv=skf, scoring='recall')

print("Recall por fold:", scores)
print("Recall médio:", scores.mean())
print("Desvio padrão:", scores.std())


Recall por fold: [0.82887701 0.81550802 0.76470588 0.81769437 0.77807487]
Recall médio: 0.800972029074852
Desvio padrão: 0.024936450907165515


## Conclusão da Etapa 2 e Tabela Comparativa de modelos

| Modelo | Recall (classe 1) | Precision (classe 1) | F1-score | Accuracy |
|---|---|---|---|---|
| Regressão Logística (balanced) | **0,80** | 0,51 | 0,62 | 0,75 |
| Random Forest (balanced) | 0,64 | 0,57 | 0,60 | 0,78 |
| Árvore de Decisão (balanced) | 0,55 | 0,51 | 0,53 | 0,74 |
| MLP (sem balanceamento) | 0,51 | 0,57 | 0,54 | 0,77 |
| MLP (SMOTE) | 0,54 | 0,49 | 0,52 | 0,73 |

Foi testado Regressão Logística, Árvore de Decisão, Random Forest e MLP (com
e sem balanceamento). A Regressão Logística com class_weight='balanced'
venceu em recall por boa margem, e a validação cruzada confirmou que o
resultado é estável.

Modelos mais complexos (árvore, ensemble, rede neural) não superaram o
baseline linear. plausível dado que a EDA já mostrava relações bem
diretas entre variáveis como Contract e tenure com o churn, favorecendo
um modelo linear.

**Modelo campeão: Regressão Logística com class_weight='balanced'**
(validado com recall médio de 0,80 ± 0,025 em 5-fold cross-validation)

In [31]:
import joblib

joblib.dump(lr_model, '../models/churn_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

['../models/scaler.pkl']

In [32]:
loaded_model = joblib.load('../models/churn_model.pkl')
loaded_scaler = joblib.load('../models/scaler.pkl')

print(loaded_model)
print(loaded_model.coef_)  # os pesos que o modelo aprendeu pra cada variável
print(loaded_scaler.mean_)  # a média que o scaler guardou de cada variável

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=1109)
[[ 0.00234289  0.09858769  0.00786063 -0.06922029 -1.31740992 -0.31283914
   0.03336583  0.1586739  -0.26887912 -0.08198094 -0.04191515 -0.21194016
  -0.0274038  -0.03224924 -0.60581631  0.16425351  0.05518836  0.72987905
   0.60889035]]
[5.06744764e-01 1.61519347e-01 4.76038339e-01 2.96769613e-01
 3.23668797e+01 9.04330848e-01 9.40539581e-01 8.75221867e-01
 7.93042244e-01 9.06283280e-01 9.04508342e-01 7.97657082e-01
 9.81185659e-01 9.87930422e-01 6.88853390e-01 5.88214412e-01
 1.57720980e+00 6.47231807e+01 2.27444901e+03]
